In [1]:
from models import NeuralNet
from data import get_dataloader, DataPreprocessor, DataSplitter
from utils import load_config
import pandas as pd
import wandb
import pyreadr
import lightning as pl
from lightning.pytorch.loggers import WandbLogger

/home/cjs/miniconda3/envs/idp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
%load_ext autoreload
%autoreload 2

## Data Loading & Preprocessing

In [8]:
r_data = pyreadr.read_r("../../data/raw/features_all_windows_combined.rds")
df = pd.DataFrame(next(iter(r_data.values())))
config = load_config("../../config.yaml")

In [ ]:
train_df, val_df, test_df = DataSplitter(test_size=0.15, val_size=0.15, random_state=42).group_split(df, config["dataset"].get("groupsplit_column", "user_id"))

preprocessor = DataPreprocessor(config)
train_transformed = preprocessor.fit_transform(train_df)
val_transformed = preprocessor.transform(val_df)
test_transformed = preprocessor.transform(test_df)
# TODO: analyze dataset: which columns have NaN and why? Does drop inside transform do anything unwanted?

train_dataloader = get_dataloader(train_transformed, config, batch_size=32, shuffle=False, num_workers=4)
val_dataloader = get_dataloader(val_transformed, config, batch_size=32, shuffle=False, num_workers=4)
test_dataloader = get_dataloader(val_transformed, config, batch_size=32, shuffle=False, num_workers=4)

BEFORE
       user_id          start_time            end_time      time_bin  \
0         1000 2020-05-21 11:40:01 2020-05-21 12:59:40          Noon   
1         1000 2020-05-21 13:11:15 2020-05-21 13:39:16     Afternoon   
2         1000 2020-05-21 14:08:38 2020-05-21 15:39:39     Afternoon   
3         1000 2020-05-21 15:53:49 2020-05-21 16:03:49     Afternoon   
4         1000 2020-05-21 16:15:18 2020-05-21 16:25:18     Afternoon   
...        ...                 ...                 ...           ...   
58484      999 2020-06-12 07:01:46 2020-06-12 07:11:46       Morning   
58485      999 2020-06-12 12:51:57 2020-06-12 13:01:57          Noon   
58486      999 2020-06-12 14:44:38 2020-06-12 14:54:38     Afternoon   
58487      999 2020-06-12 15:19:28 2020-06-12 15:29:28     Afternoon   
58488      999 2020-08-14 08:30:14 2020-08-14 08:40:14  Late Morning   

       gps_count  gps_first_latitude  gps_first_longitude gps_landuse_type  \
0            0.0                 NaN              

AttributeError: 'numpy.ndarray' object has no attribute 'drop'

In [ ]:
print(train_)

       user_id          start_time            end_time      time_bin  \
0         1000 2020-05-21 11:40:01 2020-05-21 12:59:40          Noon   
1         1000 2020-05-21 13:11:15 2020-05-21 13:39:16     Afternoon   
2         1000 2020-05-21 14:08:38 2020-05-21 15:39:39     Afternoon   
3         1000 2020-05-21 15:53:49 2020-05-21 16:03:49     Afternoon   
4         1000 2020-05-21 16:15:18 2020-05-21 16:25:18     Afternoon   
...        ...                 ...                 ...           ...   
58484      999 2020-06-12 07:01:46 2020-06-12 07:11:46       Morning   
58485      999 2020-06-12 12:51:57 2020-06-12 13:01:57          Noon   
58486      999 2020-06-12 14:44:38 2020-06-12 14:54:38     Afternoon   
58487      999 2020-06-12 15:19:28 2020-06-12 15:29:28     Afternoon   
58488      999 2020-08-14 08:30:14 2020-08-14 08:40:14  Late Morning   

       gps_count  gps_first_latitude  gps_first_longitude gps_landuse_type  \
0            0.0                 NaN                  NaN

## Model Training

In [ ]:
wandb_logger = WandbLogger(project=config.get("project", "situation-prediction"), name="nn_training")
model = NeuralNet(config, wandb_logger=wandb_logger)
trainer = pl.Trainer(max_epochs=config.get("max_epochs", 100),
                     logger=[wandb_logger],
                     precision=config.get("precision", "bf16"))
trainer.fit(model, 
            train_dataloader=train_dataloader,
            val_dataloader=val_dataloader)